In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / "notebooks").is_dir() else Path.cwd().parent

protein_path = ROOT / "data/processed/human_proteins_with_string.parquet"

protein_df = pd.read_parquet(protein_path).reset_index(drop=True)

print("Proteins loaded:", len(protein_df))

Proteins loaded: 18841


In [2]:
# 20 standard amino acids
AA_ALPHABET = list("ACDEFGHIKLMNPQRSTVWY")

# node_index will be the permanent row/node mapping for the first GNN graph
feature_df = protein_df[
    ["uniprot_id", "string_id", "sequence", "sequence_length"]
].copy()

feature_df.insert(0, "node_index", range(len(feature_df)))

# Count each standard amino acid in every sequence
aa_counts = pd.DataFrame(
    {
        aa: feature_df["sequence"].str.count(aa)
        for aa in AA_ALPHABET
    }
)

valid_residue_count = aa_counts.sum(axis=1)

# Amino-acid composition: each aa_* feature lies between 0 and 1
for aa in AA_ALPHABET:
    feature_df[f"aa_{aa}"] = aa_counts[aa] / valid_residue_count

feature_df["unknown_residue_count"] = (
    feature_df["sequence_length"] - valid_residue_count
)

feature_df["unknown_residue_fraction"] = (
    feature_df["unknown_residue_count"] / feature_df["sequence_length"]
)

# Sequence length is highly skewed, so log transform it
feature_df["log_sequence_length"] = np.log1p(feature_df["sequence_length"])

feature_df.head()

,node_index,uniprot_id,string_id,sequence,sequence_length,aa_A,aa_C,aa_D,aa_E,aa_F,...,aa_Q,aa_R,aa_S,aa_T,aa_V,aa_W,aa_Y,unknown_residue_count,unknown_residue_fraction,log_sequence_length
0,0,A0A096LP01,9606.ENSP00000485316,MYRNEFTAWYRRMSVVYGIGTWSVLGSLLYYSRTMAKSSVDQKDGS...,95,0.031579,0.000000,0.031579,0.084211,0.031579,...,0.010526,0.052632,0.115789,0.084211,0.094737,0.042105,0.084211,0,0.0,4.564348
1,1,A0A0B4J2F0,9606.ENSP00000484893,MFRRLTFAQLLFATVLGIAGGVYIFQPVFEQYAKDQKELKEKMQLV...,54,0.074074,0.000000,0.018519,0.111111,0.092593,...,0.111111,0.037037,0.037037,0.037037,0.074074,0.000000,0.037037,0,0.0,4.007333
2,2,A0A0U1RRE5,9606.ENSP00000489583,MGDQPCASGRSTLPPGNAREAKPPKKRCLLAPRWDYPEGTPNGGST...,68,0.102941,0.029412,0.029412,0.044118,0.000000,...,0.014706,0.058824,0.088235,0.058824,0.000000,0.014706,0.014706,0,0.0,4.234107
3,3,A0AV02,9606.ENSP00000377112,MTQMSQVQELFHEAAQQDALAQPQPWWKTQLFMWEPVLFGTWDGVF...,714,0.067227,0.019608,0.026611,0.051821,0.058824,...,0.051821,0.036415,0.091036,0.051821,0.077031,0.019608,0.028011,0,0.0,6.572283
4,4,A0AV96,9606.ENSP00000295971,MTAEDSTAAMSSDSAAGSSAKVPEGVAGAPNEAALLALMERTGYSM...,593,0.143339,0.013491,0.037099,0.067454,0.026981,...,0.033727,0.057336,0.042159,0.040472,0.075885,0.005059,0.059022,0,0.0,6.386879


In [3]:
composition_columns = [f"aa_{aa}" for aa in AA_ALPHABET]
composition_sum = feature_df[composition_columns].sum(axis=1)

print("Feature table shape:", feature_df.shape)
print("Node indices unique:", feature_df["node_index"].is_unique)

print("\nAmino-acid composition sum:")
print(composition_sum.describe())

print("\nUnknown residue fraction:")
print(feature_df["unknown_residue_fraction"].describe())

print("\nSequence length features:")
print(feature_df[["sequence_length", "log_sequence_length"]].describe())

Feature table shape: (18841, 28)
Node indices unique: True

Amino-acid composition sum:
count    1.884100e+04
mean     1.000000e+00
std      1.262174e-16
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
dtype: float64

Unknown residue fraction:
count    18841.000000
mean         0.000007
std          0.000275
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          0.026247
Name: unknown_residue_fraction, dtype: float64

Sequence length features:
       sequence_length  log_sequence_length
count     18841.000000         18841.000000
mean        583.779948             6.079402
std         613.670230             0.747010
min          24.000000             3.218876
25%         275.000000             5.620401
50%         435.000000             6.077642
75%         697.000000             6.548219
max       34350.000000            10.444386


In [4]:
output_path = ROOT / "data/processed/protein_sequence_features_aac.parquet"

feature_df.to_parquet(output_path, index=False)

print("Saved:", output_path)
print(f"File size: {output_path.stat().st_size / 1024**2:.2f} MB")

Saved: c:\Users\rajdeep\Desktop\Capstone Project_GNN\data\processed\protein_sequence_features_aac.parquet
File size: 12.84 MB
